Prepare instance to calculate the distances between every nodes

***The `euclid_distance` stores the multi dimension array for the distances between node 0-0, 0-1, 0-2, ...; 1-0, 1-1, 1-2, ...***

In [1]:
import math
from pprint import pformat

class Instance:
    def __init__(self):
        self.nodes = []
        self.euclid_distances = []
    def distance(self, u, v):
        return math.sqrt((u[0]-v[0])**2 + (u[1]-v[1])**2)
    def calculate_all_distances_between_nodes(self):
        for i in range(len(self.nodes)):
            current_node_distances = []
            for j in range(len(self.nodes)):
                # if not i == j:
                    current_node_distances.append(self.distance(self.nodes[i], self.nodes[j]))
            self.euclid_distances.append(current_node_distances)

Read data

In [2]:
myMap = Instance()
with open("att48.tsp", "r") as f:
    c = 0
    for line in f:
        if c < 5:
            if line == "" or line[0].isalpha():
                continue
            c += 1
            node, x, y = line.split()
            node = int(node)
            x = float(x)
            y = float(y)
            
            myMap.nodes.append((x, y)) 

myMap.calculate_all_distances_between_nodes()
print(f"Map nodes: {myMap.nodes}")
print(f"Distances between nodes: \n {pformat(myMap.euclid_distances, width=150)}")

Map nodes: [(6734.0, 1453.0), (2233.0, 10.0), (5530.0, 1424.0), (401.0, 841.0), (3082.0, 1644.0)]
Distances between nodes: 
 [[0.0, 4726.653149957166, 1204.3492018513568, 6362.502102160753, 3656.991249647721],
 [4726.653149957166, 0.0, 3587.4231699090087, 2011.6622479929379, 1841.400825458705],
 [1204.3492018513568, 3587.4231699090087, 0.0, 5162.02770236658, 2457.865740841025],
 [6362.502102160753, 2011.6622479929379, 5162.02770236658, 0.0, 2798.672899786611],
 [3656.991249647721, 1841.400825458705, 2457.865740841025, 2798.672899786611, 0.0]]


tracking resources wrapper

In [3]:
import time
import tracemalloc
from functools import wraps

def profile_performance(func):
    """A decorator that prints the time and memory usage of a function."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        tracemalloc.start()
        start_time = time.perf_counter()
        
        # Execute the actual function
        result = func(*args, **kwargs)
        
        end_time = time.perf_counter()
        current_mem, peak_mem = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        
        print(f"--- Profiling: {func.__name__} ---")
        print(f"Execution time : {end_time - start_time:.4f} seconds")
        print(f"Peak memory    : {peak_mem / 10**6:.4f} MB")
        print("-" * 35)
        
        return result
    return wrapper

## 1. No brainer

In [4]:
@profile_performance
def no_brainer_greedy():
    no_brainer_distance = 0

    for i in range(len(myMap.nodes)):
        next_node = (i + 1)%len(myMap.nodes)
        no_brainer_distance += myMap.euclid_distances[i][next_node]
        
    print(no_brainer_distance)
    
no_brainer_greedy()

19931.768171667085
--- Profiling: no_brainer_greedy ---
Execution time : 0.0003 seconds
Peak memory    : 0.0027 MB
-----------------------------------


## 2. Nearest Neighbour
node based

In [5]:
@profile_performance
def nearest_neighbour_greedy():
    visited = [False] * len(myMap.nodes)
    nn_distance = 0
    tour_list = []

    # Initiating
    start = 0
    visited[start] = True
    tour_list.append(start)
    current = start

    for i in range(len(myMap.euclid_distances)-1):
        shortest_distance = float("inf")
        nearest_index = current
        for j in range(len(myMap.euclid_distances[current])):
            # if myMap.euclid_distances[current][j] < shortest_distance and not (myMap.euclid_distances[current][j] == 0) and not j in tour_list:
            if myMap.euclid_distances[current][j] < shortest_distance and not visited[j]:
                shortest_distance = myMap.euclid_distances[current][j]
                nearest_index = j
                
        nn_distance += shortest_distance
        current = nearest_index
        visited[current] = True
        tour_list.append(current)
        
    nn_distance += myMap.euclid_distances[current][start]
    tour_list.append(start)

    print("Tour: ", tour_list, " with distance of: ", nn_distance)
    
nearest_neighbour_greedy()

Tour:  [0, 2, 4, 1, 3, 0]  with distance of:  13877.780118304778
--- Profiling: nearest_neighbour_greedy ---
Execution time : 0.0010 seconds
Peak memory    : 0.1511 MB
-----------------------------------


## 3. Repeated Nearest Neigbour
fixed the problems of `Nearest Neighbour`:
    - when returning back to the starting point, it may have to go a very long distance
-> The nearest also depends on how we choose the starting point

**Repeated Nearest Neighbor** test every single node as the starting node and then to find the shortest route

In [6]:
@profile_performance
def repeated_nearest_neighbour():
    all_nn_distances = []
    shortest = float("inf")
    for k in range(len(myMap.nodes)):
        start = k
        visited = [False] * len(myMap.nodes)
        nn_distance = 0
        tour_list = []
        
        visited[start] = True
        tour_list.append(start)
        current = start
        
        for i in range(len(myMap.euclid_distances) - 1):
            shortest_distance = float("inf")
            nearest_index = current
            for j in range(len(myMap.euclid_distances[current])):
                if myMap.euclid_distances[current][j] < shortest_distance and not visited[j]:
                    shortest_distance = myMap.euclid_distances[current][j]
                    nearest_index = j
                    
            nn_distance += shortest_distance
            current = nearest_index
            visited[current] = True
            tour_list.append(current)
            
        nn_distance += myMap.euclid_distances[current][start]
        tour_list.append(start)

        print("Tour: ", tour_list, " with distance of: ", nn_distance)
        all_nn_distances.append({"tour": tour_list, "distance": nn_distance})
        if nn_distance < shortest:
            shortest = nn_distance

    print(f"The shortest distance: {shortest}")
    
repeated_nearest_neighbour()

Tour:  [0, 2, 4, 1, 3, 0]  with distance of:  13877.780118304778
Tour:  [1, 4, 2, 0, 3, 1]  with distance of:  13877.780118304776
Tour:  [2, 0, 4, 1, 3, 2]  with distance of:  13876.431227317302
Tour:  [3, 1, 4, 2, 0, 3]  with distance of:  13877.780118304778
Tour:  [4, 1, 3, 2, 0, 4]  with distance of:  13876.4312273173
The shortest distance: 13876.4312273173
--- Profiling: repeated_nearest_neighbour ---
Execution time : 0.0025 seconds
Peak memory    : 0.1528 MB
-----------------------------------


## 4. Cheapest Insertion

- insertion-based
- imagine having an elastic band, representing our route going through all of outer nodes
    - there will be lots of nodes that we haven't passed
    - you take that elastic band and hook it to each non-passed nodes single by single one
    - each time you hook that band, it's get stretched out a little bit
    - **RULE**: You calculate the sum
- **Goal**: Make the band grow little by little step by step

vì là 1 chu trình -> với `m` đỉnh thì luôn có `m` cách để chèn 1 điểm mới vào

delta (phần band bị stretched) khi chèn `k` vào giữa `i` và `j` sẽ được tính bằng công thức: `D(i, k) + D(k, j) - D(i, j)` (thay quãng đường đi trực tiếp từ `i` đến `j` bằng 2 quãng đường đi từ `i` đến `k` rồi từ `k` đến `j`)

**wrong version**

instead of insert the best node at the best position, this version below insert the current scaning node into the best position

In [7]:
shortest_between_2_nodes = float("inf")
two_nodes_have_shortest = ()
for i in range(len(myMap.euclid_distances)):
    for j in range(len(myMap.euclid_distances[i])):
        if myMap.euclid_distances[i][j] < shortest_between_2_nodes and not myMap.euclid_distances[i][j] == 0:
            shortest_between_2_nodes = myMap.euclid_distances[i][j]
            two_nodes_have_shortest = (i, j)

print(shortest_between_2_nodes, two_nodes_have_shortest)

visited = [False] * len(myMap.nodes)
visited[two_nodes_have_shortest[0]] = True
visited[two_nodes_have_shortest[1]] = True
tour = [two_nodes_have_shortest[0], two_nodes_have_shortest[1]]
total_CI_distance = shortest_between_2_nodes * 2

for node_index in range(len(myMap.nodes)):
    if not visited[node_index]:
        # Here is where we stretch the band to the other nodes that's not in the tour
        best_delta = float("inf")
        best_insertion_position = -1
        best_insertion_node = -1
        for position in range(len(tour)):
            first_node = tour[position]
            second_node = tour[((position + 1 ) % len(tour))]
            print(f"Trying to insert {node_index} in between {first_node} and {second_node}")
            delta = myMap.euclid_distances[first_node][node_index] + myMap.euclid_distances[second_node][node_index] - myMap.euclid_distances[first_node][second_node]
            if delta < best_delta:
                best_delta = delta
                best_insertion_position = position + 1
                best_insertion_node = node_index
        
        tour.insert(best_insertion_position, best_insertion_node)
        total_CI_distance += best_delta
        
print(tour, total_CI_distance)

1204.3492018513568 (0, 2)
Trying to insert 1 in between 0 and 2
Trying to insert 1 in between 2 and 0
Trying to insert 3 in between 0 and 1
Trying to insert 3 in between 1 and 2
Trying to insert 3 in between 2 and 0
Trying to insert 4 in between 0 and 1
Trying to insert 4 in between 1 and 3
Trying to insert 4 in between 3 and 2
Trying to insert 4 in between 2 and 0
[0, 1, 3, 4, 2] 13199.203240429095


In [8]:
shortest_between_2_nodes = float("inf")
two_nodes_have_shortest = ()
for i in range(len(myMap.euclid_distances)):
    for j in range(len(myMap.euclid_distances[i])):
        if myMap.euclid_distances[i][j] < shortest_between_2_nodes and not myMap.euclid_distances[i][j] == 0:
            shortest_between_2_nodes = myMap.euclid_distances[i][j]
            two_nodes_have_shortest = (i, j)

print(shortest_between_2_nodes, two_nodes_have_shortest)

visited = [False] * len(myMap.nodes)
visited[two_nodes_have_shortest[0]] = True
visited[two_nodes_have_shortest[1]] = True
tour = [two_nodes_have_shortest[0], two_nodes_have_shortest[1]]
total_CI_distance = shortest_between_2_nodes * 2

# Step 1: Try to add all of the nodes into the band
while len(tour) < len(myMap.nodes):
    global_best_delta = float('inf')
    global_best_insertion_position = -1
    global_best_insertion_node = -1
    
    # Step 2: Scan all of the nodes that's not passed thru
    for node_index in range(len(myMap.nodes)):
        if not visited[node_index]:
            
            # Step 3: Trying to insert the un-passed node in between the passed nodes
            for position in range(len(tour)):
                first_node = tour[position]
                second_node = tour[((position + 1 ) % len(tour))]
                print(f"Trying to insert {node_index} in between {first_node} and {second_node}")
                delta = myMap.euclid_distances[first_node][node_index] + myMap.euclid_distances[second_node][node_index] - myMap.euclid_distances[first_node][second_node]
                if delta < global_best_delta:
                    global_best_delta = delta
                    global_best_insertion_position = position + 1
                    global_best_insertion_node = node_index
            
    tour.insert(global_best_insertion_position, global_best_insertion_node)
    total_CI_distance += global_best_delta
        
print(tour, total_CI_distance)

1204.3492018513568 (0, 2)
Trying to insert 1 in between 0 and 2
Trying to insert 1 in between 2 and 0
Trying to insert 3 in between 0 and 2
Trying to insert 3 in between 2 and 0
Trying to insert 4 in between 0 and 2
Trying to insert 4 in between 2 and 0
Trying to insert 1 in between 0 and 4
Trying to insert 1 in between 4 and 2
Trying to insert 1 in between 2 and 0
Trying to insert 3 in between 0 and 4
Trying to insert 3 in between 4 and 2
Trying to insert 3 in between 2 and 0
Trying to insert 4 in between 0 and 4
Trying to insert 4 in between 4 and 2
Trying to insert 4 in between 2 and 0
Trying to insert 1 in between 0 and 4
Trying to insert 1 in between 4 and 4
Trying to insert 1 in between 4 and 2
Trying to insert 1 in between 2 and 0
Trying to insert 3 in between 0 and 4
Trying to insert 3 in between 4 and 4
Trying to insert 3 in between 4 and 2
Trying to insert 3 in between 2 and 0
Trying to insert 4 in between 0 and 4
Trying to insert 4 in between 4 and 4
Trying to insert 4 in be

## 5. Nearest Insertion

Based on **Cheapest Insertion**

- Instead of calculating delta for ALL the nodes and ALL the positions to find which node stretch the band the least, Nearest Insertion does in 2 steps:
    - **Step 1**: Choose which node is ___physically nearest___ to the band, overlooking the stretchness the band can be
    - **Step 2**: Calculate the delta for that *nearest* node and find the position to insert it so that it's the cheapest


### Why do we need Nearest Insertion?
- **NI** is more visualizable than **CI**, it collects all of the nodes that's nearest to the band and increasingly stretching the band out. 
- **CI** can add the further nodes from the very start just because it's accidentally sitting at an existing edge


In [ ]:
shortest_between_2_nodes = float("inf")
two_nodes_have_shortest = ()
for i in range(len(myMap.euclid_distances)):
    for j in range(len(myMap.euclid_distances[i])):
        if myMap.euclid_distances[i][j] < shortest_between_2_nodes and not myMap.euclid_distances[i][j] == 0:
            shortest_between_2_nodes = myMap.euclid_distances[i][j]
            two_nodes_have_shortest = (i, j)

print(shortest_between_2_nodes, two_nodes_have_shortest)

visited = [False] * len(myMap.nodes)
tour = [two_nodes_have_shortest[0], two_nodes_have_shortest[1]]
visited[two_nodes_have_shortest[0]] = True
visited[two_nodes_have_shortest[1]] = True
total_NI_distance = shortest_between_2_nodes * 2

while len(tour) < len(myMap.nodes):
    shortest_distance_to_tour = float("inf")
    next_node = -1
    for i in range(len(myMap.euclid_distances)):
        if not visited[i]:
            for j in (tour):
                if shortest_distance_to_tour > myMap.euclid_distances[i][j] and (myMap.euclid_distances[i][j] != 0):
                    shortest_distance_to_tour = myMap.euclid_distances[i][j]
                    next_node = i
                
    # Insert the next node into gaps between existing nodes in tour
    best_delta = float("inf")
    best_insertion_position = -1
    for i in range(len(tour)):
        first_node = tour[i]
        second_node = tour[(i+1)%len(tour)]
        
        delta = myMap.euclid_distances[first_node][next_node] + myMap.euclid_distances[next_node][second_node] - myMap.euclid_distances[first_node][second_node]
        if delta < best_delta:
            best_delta = delta
            best_insertion_position = i

    tour.insert(best_insertion_position, next_node)
    total_NI_distance += best_delta

print(tour, total_NI_distance)

1204.3492018513568 (0, 2)
[1, 1, 4, 0, 2] 10230.268918108253
